# Análise do dataset

In [5]:
# Definindo função para colocar título
def titulo(txt):
    print("\n" + "-"*60)
    print(txt)
    print("-"*60)   

In [6]:
import pandas as pd

df = pd.read_csv("./healthcare_patient_journey.csv")

import pandas as pd

titulo("DATASET - PRIMEIRAS LINHAS")
display(df.head())

titulo("TIPOS DE DADOS")
display(df.dtypes.to_frame(name="Tipo"))

titulo("ESTATÍSTICAS DESCRITIVAS")
display(df.describe())

titulo("DIMENSÕES DO DATASET")
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")


------------------------------------------------------------
DATASET - PRIMEIRAS LINHAS
------------------------------------------------------------


,patient_id,age,gender,chronic_condition,admission_type,department,wait_time_min,length_of_stay_days,procedures_count,medication_count,complications,discharge_status,readmitted_30d,total_cost_€,satisfaction_score
0,1,69,male,0,scheduled,Neurology,41,2,0,3,1,referred,1,1440,2
1,2,38,male,0,emergency,Oncology,17,3,1,2,0,recovered,0,2060,3
2,3,81,male,0,scheduled,Neurology,40,2,3,2,0,recovered,0,2110,3
3,4,67,female,1,emergency,ER,7,4,5,9,0,recovered,0,4070,3
4,5,88,male,1,emergency,Cardiology,34,3,7,5,0,recovered,1,3800,3



------------------------------------------------------------
TIPOS DE DADOS
------------------------------------------------------------


,Tipo
patient_id,int64
age,int64
gender,object
chronic_condition,int64
admission_type,object
department,object
wait_time_min,int64
length_of_stay_days,int64
procedures_count,int64
medication_count,int64



------------------------------------------------------------
ESTATÍSTICAS DESCRITIVAS
------------------------------------------------------------


,patient_id,age,chronic_condition,wait_time_min,length_of_stay_days,procedures_count,medication_count,complications,readmitted_30d,total_cost_€,satisfaction_score
count,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,1500.500000,53.051333,0.405667,33.326333,3.535000,2.435667,3.404667,0.160333,0.234667,2772.040000,2.803000
std,866.169729,20.704901,0.491102,17.111559,2.024881,1.595842,1.890883,0.366976,0.423861,1086.215721,0.838506
min,1.000000,18.000000,0.000000,5.000000,1.000000,0.000000,0.000000,0.000000,0.000000,750.000000,1.000000
25%,750.750000,35.000000,0.000000,20.750000,2.000000,1.000000,2.000000,0.000000,0.000000,1900.000000,2.000000
50%,1500.500000,53.000000,0.000000,33.000000,3.000000,2.000000,3.000000,0.000000,0.000000,2670.000000,3.000000
75%,2250.250000,71.000000,1.000000,45.000000,5.000000,3.000000,5.000000,0.000000,0.000000,3490.000000,3.000000
max,3000.000000,89.000000,1.000000,103.000000,11.000000,10.000000,11.000000,1.000000,1.000000,6610.000000,5.000000



------------------------------------------------------------
DIMENSÕES DO DATASET
------------------------------------------------------------
Linhas: 3000
Colunas: 15


# Tratamento dos Dados

In [7]:
# Investigando se há alguma patologia nos dados

# Verificando campos que tem valores nulos
titulo("VERIFICAÇÃO DE VALORES NULOS")
print(df.isnull().any())


# Verificando se há IDs duplicados na base (que deverião ser únicos)
titulo("VERIFICAÇÃO DE IDS DUPLICADOS")
ids_duplicados = df['patient_id'].duplicated()
print(f"Quantidade de IDs duplicados: {ids_duplicados.sum()}")


# Verificando se há valores não numéricos em campos numéricos
titulo("VERIFICAÇÃO DE VALORES NUMÉRICOS")
colunas_numericas = ["age", "chronic_condition", "wait_time_min", "length_of_stay_days", "procedures_count", "medication_count", "complications", "readmitted_30d", "total_cost_€", "satisfaction_score"]

for coluna in colunas_numericas:
    invalidos = pd.to_numeric(df[coluna], errors='coerce').isna()
    print(f"Coluna: {coluna} -> {invalidos.sum()} valores não numéricos")


# Verificando se há valores negativos que podem corromper a base
titulo("VERIFICAÇÃO DE VALORES NEGATIVOS")
for coluna in colunas_numericas:
    negativos = (df[coluna] < 0)
    print(f"Coluna: {coluna} -> {negativos.sum()} valores negativos")


# Verificando se há valores diferentes de 0 ou 1 em campos binários
titulo("VERIFICAÇÃO DE CAMPOS BINÁRIOS")
colunas_binarias = ["chronic_condition", "complications", "readmitted_30d"]

for coluna in colunas_binarias:
    invalidos = (df[coluna] != 0) & (df[coluna] != 1)
    print(f"Coluna: {coluna} -> {invalidos.sum()} valores inválidos")


# Verificando como estão os dados no campo ´total_cost´
titulo("VERIFICAÇÃO DO CAMPO total_cost")
print(df['total_cost_€'])


# Verificando a padronização dos campos categóricos
titulo("VERIFICAÇÃO DE CAMPOS CATEGÓRICOS")

print(f"Gêneros: {df['gender'].unique()}")
print(f"Tipos de admissão: {df['admission_type'].unique()}")
print(f"Departamentos: {df['department'].unique()}")
print(f"Status de alta: {df['discharge_status'].unique()}")


# Verificando a maior e menor idade
titulo("VERIFICAÇÃO DOS LIMITES DAS IDADES")
print(f"Idade mínima: {df['age'].min()}")
print(f"Idade máxima: {df['age'].max()}")



------------------------------------------------------------
VERIFICAÇÃO DE VALORES NULOS
------------------------------------------------------------
patient_id             False
age                    False
gender                 False
chronic_condition      False
admission_type         False
department             False
wait_time_min          False
length_of_stay_days    False
procedures_count       False
medication_count       False
complications          False
discharge_status       False
readmitted_30d         False
total_cost_€           False
satisfaction_score     False
dtype: bool

------------------------------------------------------------
VERIFICAÇÃO DE IDS DUPLICADOS
------------------------------------------------------------
Quantidade de IDs duplicados: 0

------------------------------------------------------------
VERIFICAÇÃO DE VALORES NUMÉRICOS
------------------------------------------------------------
Coluna: age -> 0 valores não numéricos
Coluna: chronic_condi

In [8]:
# Conversão do euro para dólar
dolar = 1.16
df['total_cost_€'] = df['total_cost_€'] * dolar

# Deixando os nomes dos departamentos em minúsculo
df['department'] = df['department'].str.lower()

In [9]:
# Renomeação das colunas
df.rename(columns={
    "patient_id": "id_paciente",
    "age": "idade",
    "gender": "genero",
    "chronic_condition": "condicao_cronica",
    "admission_type": "tipo_admissao",
    "department": "departamento",
    "wait_time_min": "tempo_espera_min",
    "length_of_stay_days": "duracao_estadia_dias",
    "procedures_count": "qtd_procedimentos",
    "medication_count": "qtd_medicamentos",
    "complications": "complicacoes",
    "discharge_status": "status_alta",
    "readmitted_30d": "retorno_apos_30d",
    "total_cost_€": "custo_total_dolares",
    "satisfaction_score": "nivel_satisfacao"
}, inplace=True)

In [10]:
# Tradução dos valores categóricos

df['genero'] = df['genero'].replace({
    'male': 'masculino',
    'female': 'feminino'
})

df['tipo_admissao'] = df['tipo_admissao'].replace({
    'scheduled': 'agendada',
    'emergency': 'emergência'
})

df['departamento'] = df['departamento'].replace({
    'neurology': 'neurologia',
    'oncology': 'oncologia',
    'er': 'pronto socorro',
    'cardiology': 'cardiologia',
    'polyclinic': 'policlínica'
})

df['status_alta'] = df['status_alta'].replace({
    'referred': 'encaminhado',
    'recovered': 'recuperado'
})

%md
# Resultado da Análise e do Tratamento de Colunas

Ao analisar as 15 colunas já vindas do dataset, percebeu-se que não havia nenhum tipo de patologia nos dados. Todos os campos estavam devidamente tratados e com seus devidos tipos:
- Todas as colunas numéricas tinham apenas dados numéricos
- Colunas binárias (de 0 ou 1) não tinham dados diferentes desses dois números
- Não haviam IDs duplicados
- Nenhum tipo de dado nulo dentre os 3000 dados presentes na base
- Não há falta de padronização dentre as colunas categóricas -> Tudo em minúsculo, com exceção do campo `departament` que foi tratado para todos os dados serem minúsculos
- Idades presentes no campo `age` entre 18 e 89 anos, que são dados coerentes e estáveis dentro da base

## Renomeação de Colunas

As colunas estavam de fácil entendimento, porém estavam em inglês, e como nosso dashboard será feito em português decidimos renomeá-las para facilitar a interpretação dos dados e tornar a visualização mais intuitiva para os usuários finais.

Segue abaixo a renomeação realizada:

- `patient_id` → `id_paciente`
- `age` → `idade`
- `gender` → `genero`
- `chronic_condition` → `condicao_cronica`
- `admission_type` → `tipo_admissao`
- `department` → `departamento`
- `wait_time_min` → `tempo_espera_min`
- `length_of_stay_days` → `duracao_estadia_dias`
- `procedures_count` → `qtd_procedimentos`
- `medication_count` → `qtd_medicamentos`
- `complications` → `complicacoes`
- `discharge_status` → `status_alta`
- `readmitted_30d` → `retorno_apos_30d`
- `total_cost_€` → `custo_total_dolares`
- `satisfaction_score` → `nivel_satisfacao`

## Conversão do Campo `custo_total_dolares`
Ao analisar a base e pensarmos na elaboração do dashboard, decidimos converter do euro para o dólar pelo fato de a moeda oficial do dashboard ser o dólar ($).

Levamos em consideração o valor do dólar no dia **22/05/2026**

1 euro (€) = 1,16 dólars americanos ($)

In [11]:
# Subindo base para o spark
# spark.createDataFrame(df).write.mode("overwrite").saveAsTable("df_healthcare_dataset")

# KPIs Definidos

#### Tempo de Espera
- Tempo Médio de Espera Geral
- Desvio padrão do tempo de espera
- Tempo Médio de Espera p/Departamento
- Tempo de Espera p/Tipo de Admissão

#### Período de Estadia
- Período Médio de Estadia Geral
- Desvio Padrão da Estadia
- Período Médio de Estadia p/Departamento
- Nível de Satisfação p/Período de Estadia
- Estadia Média por Readmissão

#### Quantidade de Pacientes
- Quantidade Total de Pacientes
- Quantidade de Pacientes p/Departamento
- Quantidade de Pacientes p/Tipo de Admissão
- Quantidade de Pacientes com Doenças Crônicas
- Pacientes de Alto Custo (Top 25% dos Custos Totais)

#### Procedimentos e Medicamentos
- Total Geral de Procedimentos
- Quantidade de Procedimentos p/Departamento
- Quantidade de Medicamentos x Preço x Departamento
- Quantidade de Medicamentos por Quantidade de Procedimentos

#### Valor Gasto
- Preço Médio Geral
- Desvio Padrão do Custo
- Preço Médio p/Departamento
- Preço por Satisfação
- Custo Médio por Nível de Satisfação

#### Nível de Satisfação
- Nível de Satisfação Geral
- Desvio Padrão da Satisfação
- Nível de Satisfação p/Departamento
- Nível de Satisfação p/Tempo de Espera
- Satisfação p/Gênero

#### Pacientes com Complicações
- Taxa de Complicações
- Médias por Complicações

#### Pacientes com Condições Crônicas
- Médias por Condição Crônica

#### Readmissão
- Médias por Readmissão em 30 Dias
- Readmissão por Faixa Etária
- Readmissão por Tipo de Admissão
- Readmissão por Complicações
- Readmissão por Condição Crônica

In [12]:

# Funções para mostrar Kpis
def mostrar_kpi(titulo_kpi, valor, sufixo=""):
    titulo(titulo_kpi)
    print(round(valor, 2), sufixo)

def mostrar_kpi_groupby(titulo_kpi, serie):
    titulo(titulo_kpi)
    print(serie.round(2))


# TEMPO DE ESPERA
mostrar_kpi("TEMPO MÉDIO DE ESPERA", df["tempo_espera_min"].mean(), "minutos.")
mostrar_kpi("DESVIO PADRÃO DO TEMPO DE ESPERA", df["tempo_espera_min"].std(), "minutos.")

mostrar_kpi_groupby(
    "TEMPO MÉDIO POR DEPARTAMENTO",
    df.groupby("departamento")["tempo_espera_min"].mean()
)

mostrar_kpi_groupby(
    "TEMPO MÉDIO POR TIPO DE ADMISSÃO",
    df.groupby("tipo_admissao")["tempo_espera_min"].mean()
)


# ESTADIA
mostrar_kpi("PERÍODO MÉDIO DE ESTADIA GERAL", df["duracao_estadia_dias"].mean(), "dias.")
mostrar_kpi("DESVIO PADRÃO DA ESTADIA", df["duracao_estadia_dias"].std(), "dias.")

mostrar_kpi_groupby(
    "ESTADIA MÉDIA POR DEPARTAMENTO",
    df.groupby("departamento")["duracao_estadia_dias"].mean()
)

mostrar_kpi_groupby(
    "ESTADIA MÉDIA POR READMISSÃO",
    df.groupby("retorno_apos_30d")[
        "duracao_estadia_dias"
    ].mean()
)

# PACIENTES
mostrar_kpi("QUANTIDADE TOTAL DE PACIENTES", df["id_paciente"].count(), "pacientes.")

mostrar_kpi_groupby(
    "PACIENTES POR DEPARTAMENTO",
    df.groupby("departamento")["id_paciente"].count()
)

mostrar_kpi_groupby(
    "PACIENTES POR TIPO DE ADMISSÃO",
    df.groupby("tipo_admissao").size()
)

alto_custo = df[
    df["custo_total_dolares"] > 
    df["custo_total_dolares"].quantile(0.75)
]

mostrar_kpi(
    "PACIENTES DE ALTO CUSTO",
    alto_custo.shape[0],
    "pacientes"
)


# DOENÇAS CRÔNICAS
mostrar_kpi("PACIENTES COM DOENÇA CRÔNICA", df["condicao_cronica"].sum(), "pacientes.")

mostrar_kpi_groupby(
    "MÉDIAS POR CONDIÇÃO CRÔNICA",
    df.groupby("condicao_cronica")[[
        "duracao_estadia_dias",
        "qtd_medicamentos",
        "custo_total_dolares"
    ]].mean()
)

# PROCEDIMENTOS
mostrar_kpi("TOTAL DE PROCEDIMENTOS", df["qtd_procedimentos"].sum(), "procedimentos.")

mostrar_kpi_groupby(
    "PROCEDIMENTOS POR DEPARTAMENTO",
    df.groupby("departamento")["qtd_procedimentos"].sum()
)


# CUSTOS
mostrar_kpi("PREÇO MÉDIO GERAL", df["custo_total_dolares"].mean())
mostrar_kpi("DESVIO PADRÃO DO CUSTO", df["custo_total_dolares"].std())

mostrar_kpi_groupby(
    "CUSTO MÉDIO POR DEPARTAMENTO",
    df.groupby("departamento")["custo_total_dolares"].mean()
)

mostrar_kpi_groupby(
    "CUSTO MÉDIO POR NÍVEL DE SATISFAÇÃO",
    df.groupby("nivel_satisfacao")["custo_total_dolares"].mean()
)

mostrar_kpi_groupby(
    "MEDICAMENTOS E CUSTO POR DEPARTAMENTO",
    df.groupby("departamento")[[
        "qtd_medicamentos",
        "custo_total_dolares"
    ]].mean()
)

# SATISFAÇÃO
mostrar_kpi("SATISFAÇÃO MÉDIA GERAL", df["nivel_satisfacao"].mean())
mostrar_kpi("DESVIO PADRÃO DA SATISFAÇÃO", df["nivel_satisfacao"].std())

mostrar_kpi_groupby(
    "SATISFAÇÃO POR DEPARTAMENTO",
    df.groupby("departamento")["nivel_satisfacao"].mean()
)

mostrar_kpi_groupby(
    "SATISFAÇÃO POR GÊNERO",
    df.groupby("genero")["nivel_satisfacao"].mean()
)


# FAIXA DE ESPERA
df["faixa_espera"] = pd.cut(
    df["tempo_espera_min"],
    bins=[0, 15, 30, 60, 120],
    labels=["0-15", "16-30", "31-60", "61-120"]
)

mostrar_kpi_groupby(
    "SATISFAÇÃO POR FAIXA DE ESPERA",
    df.groupby("faixa_espera")["nivel_satisfacao"].mean()
)


# FAIXA DE ESTADIA
df["faixa_estadia"] = pd.cut(
    df["duracao_estadia_dias"],
    bins=[0, 3, 7, 15],
    labels=["0-3", "4-7", "8-15"]
)

mostrar_kpi_groupby(
    "SATISFAÇÃO POR FAIXA DE ESTADIA",
    df.groupby("faixa_estadia")["nivel_satisfacao"].mean()
)

# COMPLICAÇÕES

mostrar_kpi(
    "TAXA DE COMPLICAÇÕES",
    df["complicacoes"].mean() * 100,
    "%"
)

mostrar_kpi_groupby(
    "MÉDIAS POR COMPLICAÇÕES",
    df.groupby("complicacoes")[[
        "custo_total_dolares",
        "duracao_estadia_dias",
        "nivel_satisfacao"
    ]].mean()
)

mostrar_kpi(
    "TAXA DE READMISSÃO",
        df["retorno_apos_30d"].mean() * 100,
    "%"
)

mostrar_kpi_groupby(
    "MÉDIAS POR READMISSÃO EM 30 DIAS",
    df.groupby("retorno_apos_30d")[[
        "nivel_satisfacao",
        "duracao_estadia_dias",
        "custo_total_dolares"
    ]].mean()
)

mostrar_kpi_groupby(
    "READMISSÃO POR TIPO DE ADMISSÃO",
    df.groupby("tipo_admissao")[
        "retorno_apos_30d"
    ].mean() * 100
)

mostrar_kpi_groupby(
    "READMISSÃO POR CONDIÇÃO CRÔNICA",
    df.groupby("condicao_cronica")[
        "retorno_apos_30d"
    ].mean() * 100
)

mostrar_kpi_groupby(
    "READMISSÃO POR COMPLICAÇÕES",
    df.groupby("complicacoes")[
        "retorno_apos_30d"
    ].mean() * 100
)

df["faixa_idade"] = pd.cut(
    df["idade"],
    bins=[0, 18, 30, 45, 60, 75, 120],
    labels=["0-18", "19-30", "31-45", "46-60", "61-75", "75+"]
)

mostrar_kpi_groupby(
    "READMISSÃO POR FAIXA ETÁRIA (%)",
    df.groupby("faixa_idade")["retorno_apos_30d"]
    .mean() * 100
)





------------------------------------------------------------
TEMPO MÉDIO DE ESPERA
------------------------------------------------------------
33.33 minutos.

------------------------------------------------------------
DESVIO PADRÃO DO TEMPO DE ESPERA
------------------------------------------------------------
17.11 minutos.

------------------------------------------------------------
TEMPO MÉDIO POR DEPARTAMENTO
------------------------------------------------------------
departamento
cardiologia       32.81
neurologia        32.99
oncologia         33.37
policlínica       33.77
pronto socorro    33.67
Name: tempo_espera_min, dtype: float64

------------------------------------------------------------
TEMPO MÉDIO POR TIPO DE ADMISSÃO
------------------------------------------------------------
tipo_admissao
agendada      44.34
emergência    25.48
Name: tempo_espera_min, dtype: float64

------------------------------------------------------------
PERÍODO MÉDIO DE ESTADIA GERAL
---

In [ ]:
# Adicionando colunas auxiliares para o dashboard
df['alto_custo'] = df['custo_total_dolares'] > df['custo_total_dolares'].quantile(0.75)
df['alto_custo'] = df['alto_custo'].astype(int)

# Resumo dos KPIs

## 1. Tempo de Espera
Média de 33,33 minutos com alta variação (17,11).  
Consultas agendadas têm maior espera (44,34) do que emergências (25,48).  
Principal problema: agendamentos gerando filas maiores.

---

## 2. Estadia Hospitalar
Média de 3,54 dias.  
Pacientes readmitidos ficam mais tempo internados (3,96 vs 3,40).  
Departamentos são consistentes.

---

## 3. Perfil de Pacientes
3000 pacientes no total.  
Alta proporção de emergências e pacientes crônicos (1217).  
735 pacientes são de alto custo.

---

## 4. Doenças Crônicas
Crônicos têm maior impacto:
- Mais dias internados
- Mais medicamentos
- Maior custo

---

## 5. Procedimentos
Total de 7307 procedimentos.  
Distribuição equilibrada entre departamentos.

---

## 6. Custos
Custo médio de 3215 com alta variação.  
Crônicos e baixa satisfação estão associados a maior custo.

---

## 7. Satisfação
Média baixa (2,8).  
Principal fator de impacto: tempo de espera (quanto maior, menor satisfação).

---

## 8. Complicações
Taxa de 16,03%.  
Aumenta custo, tempo de internação e reduz satisfação.

---

## 9. Readmissão
Taxa de 23,47%.  
Altamente associada a:
- Complicações (53%)
- Doença crônica (38%)

---

## 10. Idade
Maior risco entre 61–75 anos (25,93%).  
Jovens têm menor taxa (~21%).

---

## Conclusão
Principais fatores críticos:
- Doença crônica e complicações aumentam custo e readmissão
- Tempo de espera impacta diretamente a satisfação
- Operação geral está equilibrada entre departamentos